In [2]:
# Create a Meaningful Dataset
np.random.seed(42)
n = 500
data = pd.DataFrame({
 "Training_Hours": np.random.randint(0, 41, n),
 "Attendance_Rate": np.random.randint(70, 101, n),
 "Task_Completion_Rate": np.random.randint(40, 101, n),
 "Quality_Score": np.random.randint(40, 101, n),
 "KPI_Achievement": np.random.randint(35, 101, n),
 "Team_Collaboration": np.random.randint(1, 11, n),
 "Customer_Feedback": np.random.randint(1, 11, n),
 "Absence_Days": np.random.randint(0, 16, n)
})

In [3]:
# Create Meaningful Target
score = (
 data["Training_Hours"] * 0.4
 + data["Attendance_Rate"] * 0.20
 + data["Task_Completion_Rate"] * 0.25
 + data["Quality_Score"] * 0.30
 + data["KPI_Achievement"] * 0.25
 + data["Team_Collaboration"] * 2
 + data["Customer_Feedback"] * 2
 - data["Absence_Days"] * 3
)

In [4]:
# Add Some Randomness
score = score + np.random.normal(0, 15, n)
data["Performance_Status"] = np.where(
 score >= np.median(score),
 1,
 0
)
print("Dataset created successfully!")
print("Shape:", data.shape)
display(data.head())


Dataset created successfully!
Shape: (500, 9)


,Training_Hours,Attendance_Rate,Task_Completion_Rate,Quality_Score,KPI_Achievement,Team_Collaboration,Customer_Feedback,Absence_Days,Performance_Status
0,38,82,98,89,56,4,6,3,1
1,28,96,84,90,54,2,2,2,1
2,14,88,45,56,98,7,1,0,0
3,7,86,76,53,51,6,10,11,0
4,20,73,96,48,46,3,5,14,0


In [5]:
# Separate Features and Target
X = data.drop("Performance_Status", axis=1)
y = data["Performance_Status"]
print("Features:")
print(X.columns.tolist())
print("\nTarget:")
print("Performance_Status")

Features:
['Training_Hours', 'Attendance_Rate', 'Task_Completion_Rate', 'Quality_Score', 'KPI_Achievement', 'Team_Collaboration', 'Customer_Feedback', 'Absence_Days']

Target:
Performance_Status


In [6]:
# Split Dataset
X_train, X_test, y_train, y_test = train_test_split(
 X,
 y,
 test_size=0.25,
 random_state=42,
 stratify=y
)
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


Training samples: 375
Testing samples: 125


In [7]:
# Define Models
models = {
 "Logistic Regression": Pipeline([
 ("scaler", StandardScaler()),
 ("model", LogisticRegression(max_iter=1000))
 ]),
 "KNN": Pipeline([
 ("scaler", StandardScaler()),
 ("model", KNeighborsClassifier())
 ]),
 "Decision Tree": Pipeline([
 ("model", DecisionTreeClassifier(random_state=42))
 ]),
 "Random Forest": Pipeline([
 ("model", RandomForestClassifier(random_state=42))
 ]),
 "SVM": Pipeline([
 ("scaler", StandardScaler()),
 ("model", SVC())
 ])
}

In [8]:
# Define Hyperparameter Grids
param_grids = {
 "Logistic Regression": {
 "model__C": [0.01, 0.1, 1, 10]
 },
 "KNN": {
 "model__n_neighbors": [3, 5, 7, 9],
 "model__weights": ["uniform", "distance"]
 },
 "Decision Tree": {
 "model__max_depth": [2, 3, 4, 5, None],
 "model__min_samples_split": [2, 5, 10]
 },
 "Random Forest": {
 "model__n_estimators": [50, 100, 150],
 "model__max_depth": [3, 5, 8, None]
 },
 "SVM": {
 "model__C": [0.1, 1, 10],
 "model__kernel": ["linear", "rbf"]
 }
}

In [30]:
results = []

best_model = None
best_model_name = None
best_accuracy = 0
best_parameters = None

for model_name, model in models.items():

    print("\n" + "=" * 70)
    print("Training:", model_name)
    print("=" * 70)

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[model_name],
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
# Best tuned model
    tuned_model = grid.best_estimator_

    # Prediction
    y_pred = tuned_model.predict(X_test)

    # Metrics
    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred
    )

    recall = recall_score(
        y_test,
        y_pred
    )

    f1 = f1_score(
        y_test,
        y_pred
    )

    print("Best Parameters:")
    print(grid.best_params_)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Best Parameters": grid.best_params_
    })

    # Select best model
    if accuracy > best_accuracy:

        best_accuracy = accuracy
        best_model = tuned_model
        best_model_name = model_name
        best_parameters = grid.best_params_


Training: Logistic Regression
Best Parameters:
{'model__C': 1}
Accuracy : 0.8080
Precision: 0.7969
Recall   : 0.8226
F1 Score : 0.8095

Training: KNN
Best Parameters:
{'model__n_neighbors': 7, 'model__weights': 'uniform'}
Accuracy : 0.7600
Precision: 0.7424
Recall   : 0.7903
F1 Score : 0.7656

Training: Decision Tree
Best Parameters:
{'model__max_depth': 3, 'model__min_samples_split': 2}
Accuracy : 0.7520
Precision: 0.7246
Recall   : 0.8065
F1 Score : 0.7634

Training: Random Forest
Best Parameters:
{'model__max_depth': 8, 'model__n_estimators': 50}
Accuracy : 0.7440
Precision: 0.7143
Recall   : 0.8065
F1 Score : 0.7576

Training: SVM
Best Parameters:
{'model__C': 10, 'model__kernel': 'linear'}
Accuracy : 0.8080
Precision: 0.7969
Recall   : 0.8226
F1 Score : 0.8095


In [31]:
# Model Comparison
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
 by="Accuracy",
 ascending=False
)
print("=" * 80)
print("AUTOML MODEL COMPARISON")
print("=" * 80)
display(results_df)

AUTOML MODEL COMPARISON


,Model,Accuracy,Precision,Recall,F1 Score,Best Parameters
0,Logistic Regression,0.808,0.796875,0.822581,0.809524,{'model__C': 1}
4,SVM,0.808,0.796875,0.822581,0.809524,"{'model__C': 10, 'model__kernel': 'linear'}"
1,KNN,0.760,0.742424,0.790323,0.765625,"{'model__n_neighbors': 7, 'model__weights': 'u..."
2,Decision Tree,0.752,0.724638,0.806452,0.763359,"{'model__max_depth': 3, 'model__min_samples_sp..."
3,Random Forest,0.744,0.714286,0.806452,0.757576,"{'model__max_depth': 8, 'model__n_estimators':..."


In [32]:
# Best Model
print("=" * 70)
print("BEST MODEL")
print("=" * 70)
print("Model:", best_model_name)
print("Accuracy:", round(best_accuracy, 4))
print("Best Parameters:", best_parameters)

BEST MODEL
Model: Logistic Regression
Accuracy: 0.808
Best Parameters: {'model__C': 1}


In [21]:
# Classification Report
best_predictions = best_model.predict(X_test)
print("Classification Report")
print("=" * 70)
print(
 classification_report(
 y_test,
 best_predictions,
 target_names=["Needs Support", "On Track"]
 )
)

Classification Report
               precision    recall  f1-score   support

Needs Support       0.82      0.79      0.81        63
     On Track       0.80      0.82      0.81        62

     accuracy                           0.81       125
    macro avg       0.81      0.81      0.81       125
 weighted avg       0.81      0.81      0.81       125



In [22]:
# Confusion Matrix
cm = confusion_matrix(y_test, best_predictions)
print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[50 13]
 [11 51]]


In [23]:
# Save Best Model
joblib.dump(
 best_model,
 "best_employee_performance_model.pkl"
)
print("Best model saved successfully!")
print("File: best_employee_performance_model.pkl")

Best model saved successfully!
File: best_employee_performance_model.pkl


In [24]:
# Load Saved Model
loaded_model = joblib.load(
 "best_employee_performance_model.pkl"
)

In [26]:
# Predict New Employee 1
new_employee = pd.DataFrame({
 "Training_Hours": [28],
 "Attendance_Rate": [94],
 "Task_Completion_Rate": [88],
 "Quality_Score": [85],
 "KPI_Achievement": [82],
 "Team_Collaboration": [8],
 "Customer_Feedback": [8],
 "Absence_Days": [1]
})
prediction = loaded_model.predict(new_employee)
if prediction[0] == 1:
 result = "ON TRACK"
else:
 result = "NEEDS SUPPORT"
print("Employee 1 Prediction:", result)

Employee 1 Prediction: ON TRACK


In [27]:
# Predict New Employee 2
new_employee = pd.DataFrame({
 "Training_Hours": [4],
 "Attendance_Rate": [76],
 "Task_Completion_Rate": [55],
 "Quality_Score": [58],
 "KPI_Achievement": [48],
 "Team_Collaboration": [4],
 "Customer_Feedback": [5],
 "Absence_Days": [10]
})
prediction = loaded_model.predict(new_employee)
if prediction[0] == 1:
 result = "ON TRACK"
else:
 result = "NEEDS SUPPORT"
print("Employee 2 Prediction:", result)

Employee 2 Prediction: NEEDS SUPPORT
